In [1]:
# Cell 1 — Imports and load CLEANED data 
import pandas as pd
import sqlite3

# cleaned outputs
df_main = pd.read_csv('../data/cleaned/df_main.csv')
df_orders_clean = pd.read_csv('../data/cleaned/orders_clean.csv')
df_products_clean = pd.read_csv('../data/cleaned/products_clean.csv')

print(f"df_main: {df_main.shape}")
print(f"orders_clean: {df_orders_clean.shape}")
print(f"products_clean: {df_products_clean.shape}")


df_main: (110197, 28)
orders_clean: (96478, 12)
products_clean: (32951, 10)


In [2]:
# Cell 2 — Create SQLite database and load tables
conn = sqlite3.connect(':memory:')  # in-memory database, resets each session

df_main.to_sql('orders_full', conn, if_exists='replace', index=False)
df_orders_clean.to_sql('orders', conn, if_exists='replace', index=False)
df_products_clean.to_sql('products', conn, if_exists='replace', index=False)

print("3 tables loaded into SQLite")

# Verify with a simple test query
test = pd.read_sql_query("SELECT COUNT(*) as total_rows FROM orders_full", conn)
print(test)

3 tables loaded into SQLite
   total_rows
0      110197


In [3]:
# Cell 3 — Query 1: Basic SELECT with column selection
result = pd.read_sql_query("""
    SELECT order_id, customer_state, price, order_status
    FROM orders_full
    LIMIT 10
""", conn)
print(result)

                           order_id customer_state   price order_status
0  e481f51cbdc54678b7cc49136f2d6af7             SP   29.99    delivered
1  53cdb2fc8bc7dce0b6741e2150273451             BA  118.70    delivered
2  47770eb9100c2d0c44946d9cf07ec65d             GO  159.90    delivered
3  949d5b44dbf5de918fe9c16f97b45f8a             RN   45.00    delivered
4  ad21c59c0840e6cb83a9ceb5573f8159             SP   19.90    delivered
5  a4591c265e18cb1dcee52889e2d8acc3             PR  147.90    delivered
6  6514b8ad8028c9f2cc2374ded245783f             RJ   59.99    delivered
7  76c6e866289321a7c93b82b54852dc33             RS   19.90    delivered
8  e69bfb5eb88e0ed6a785585b27e16dbf             SP  149.99    delivered
9  e6ce16cb79ec1d90b1da9085a6118aeb             RJ   99.00    delivered


In [4]:
# Cell 4 — Query 2: WHERE clause — filter rows
# Business Q: Which orders had a price above ₹500?

result = pd.read_sql_query("""
    SELECT order_id, customer_state, price
    FROM orders_full
    WHERE price > 500
    ORDER BY price DESC
    LIMIT 10
""", conn)
print("Q: Top 10 highest-priced orders above ₹500")
print(result)

Q: Top 10 highest-priced orders above ₹500
                           order_id customer_state    price
0  0812eb902a67711a1cb742b3cdaa65ae             MS  6735.00
1  fefacc66af859508bf1a7934eab1e97f             ES  6729.00
2  f5136e38d1a14a4dbd87dff67da82701             SP  6499.00
3  a96610ab360d42a2e5335a3998b4718a             RJ  4799.00
4  199af31afc78c699f0dbf71fb178d4d4             SP  4690.00
5  8dbc85d1447242f3b127dda390d56e19             PB  4590.00
6  426a9742b533fc6fed17d1fd6d143d7e             SP  4399.87
7  68101694e5c5dc7330c91e1bbc36214f             MG  4099.99
8  b239ca7cd485940b31882363b52e6674             MG  4059.00
9  86c4eab1571921a6a6e248ed312f5a5a             SP  3999.90


Business Insight:
 In the  list of top 10 highest priced orders above 500 , the state MS has ranked first with highest price noted as 6735 rupees and we can also see that SP state listed 4 times in the top 10 meaning 40% of high priced orders,SP state orders high priced orders more frequently . This is what i understood, but is it true?,Is SP over represented? needs to conclude the statement "SP state orders high priced orders more" i passed.

In [6]:
# Cell 5 — Query 3: Multiple conditions with AND
# Business Q: Which high-value orders from São Paulo were delivered late?

result = pd.read_sql_query("""
    SELECT order_id, customer_state, price, delivery_delay_days
    FROM orders_full
    WHERE customer_state = 'SP'
      AND price > 300
      AND delivery_delay_days > 0
    ORDER BY delivery_delay_days DESC
    LIMIT 10
""", conn)
print("Q: High-value SP orders delivered late")
print(result)

Q: High-value SP orders delivered late
                           order_id customer_state    price  \
0  47b40429ed8cce3aee9199792275433f             SP   399.00   
1  a6a6c002fd9f0e9eb0c0148448cf8863             SP   659.99   
2  6e3a1f3ec46461756c3f620e267aa1b8             SP   329.90   
3  97f48024fcc76f1898e397ad6966e3a0             SP  1259.00   
4  2f71a7d28c0418b03cd08aed6b0a4400             SP   440.00   
5  c632f9a8a1b942e306b0335429bdb142             SP   659.80   
6  8cb5fe50fcb1a1155fa2f27857895eaf             SP   559.00   
7  a6a5d5673d42186a8b8b94ad29b59897             SP   449.90   
8  a6a5d5673d42186a8b8b94ad29b59897             SP   449.90   
9  a0bba9f8b36a29b7ee6af6df702b309d             SP   370.00   

   delivery_delay_days  
0                175.0  
1                104.0  
2                 85.0  
3                 80.0  
4                 75.0  
5                 58.0  
6                 31.0  
7                 31.0  
8                 31.0  
9                

Business Insight:
Even though SP state has more orders but the no.of delay delivered days are extreme , recorder bighest delayed number of days are 175 i.e, more than 4 months.This might be a problem in logistics. This effects the customers long time. Platform might loose customers more ,if it is ignored or not solved.

In [7]:
# Cell 6 — Query 4: COUNT and basic aggregation
# Business Q: How many orders came from each state? (Day 2 recap, now in SQL)

result = pd.read_sql_query("""
    SELECT customer_state, COUNT(DISTINCT order_id) as total_orders
    FROM orders_full
    GROUP BY customer_state
    ORDER BY total_orders DESC
    LIMIT 10
""", conn)
print("Q: Orders by state")
print(result)

Q: Orders by state
  customer_state  total_orders
0             SP         40501
1             RJ         12350
2             MG         11354
3             RS          5345
4             PR          4923
5             SC          3546
6             BA          3256
7             DF          2080
8             ES          1995
9             GO          1957


In [8]:
# Cell 7 — Business Question A
# Q: What is the total revenue and average order value by state?

result = pd.read_sql_query("""
    SELECT 
        customer_state,
        COUNT(DISTINCT order_id) as total_orders,
        ROUND(SUM(price), 2) as total_revenue,
        ROUND(AVG(price), 2) as avg_item_price
    FROM orders_full
    GROUP BY customer_state
    ORDER BY total_revenue DESC
    LIMIT 10
""", conn)
print("Q: Revenue by state")
print(result)

Q: Revenue by state
  customer_state  total_orders  total_revenue  avg_item_price
0             SP         40501     5067633.16          109.10
1             RJ         12350     1759651.13          124.42
2             MG         11354     1552481.83          120.20
3             RS          5345      728897.47          118.83
4             PR          4923      666063.51          117.91
5             SC          3546      507012.13          123.75
6             BA          3256      493584.14          134.02
7             DF          2080      296498.41          125.90
8             GO          1957      282836.70          124.21
9             ES          1995      268643.45          120.74


In [9]:
# Verify Explanation 1 — check order count vs avg price relationship
result = pd.read_sql_query("""
    SELECT 
        customer_state,
        COUNT(DISTINCT order_id) as total_orders,
        ROUND(AVG(price), 2) as avg_item_price
    FROM orders_full
    GROUP BY customer_state
    HAVING total_orders > 50
    ORDER BY total_orders DESC
""", conn)
print(result)

   customer_state  total_orders  avg_item_price
0              SP         40501          109.10
1              RJ         12350          124.42
2              MG         11354          120.20
3              RS          5345          118.83
4              PR          4923          117.91
5              SC          3546          123.75
6              BA          3256          134.02
7              DF          2080          125.90
8              ES          1995          120.74
9              GO          1957          124.21
10             PE          1593          144.27
11             CE          1279          154.11
12             PA           946          165.53
13             MT           886          146.76
14             MA           717          146.26
15             MS           701          142.33
16             PB           517          192.13
17             PI           476          161.99
18             RN           474          157.59
19             AL           397         

In [10]:
# Verify Explanation 3 — check top categories in SP vs a high avg-price state
result = pd.read_sql_query("""
    SELECT 
        customer_state,
        product_category_name_english,
        COUNT(DISTINCT order_id) as orders,
        ROUND(AVG(price), 2) as avg_price
    FROM orders_full
    WHERE customer_state IN ('SP', 'RR')
    GROUP BY customer_state, product_category_name_english
    ORDER BY customer_state, orders DESC
    LIMIT 20
""", conn)
print(result)

   customer_state product_category_name_english  orders  avg_price
0              RR                sports_leisure       6     168.82
1              RR                 health_beauty       5      85.12
2              RR                     telephony       4      25.11
3              RR         computers_accessories       4     109.45
4              RR               home_appliances       3     318.06
5              RR               furniture_decor       3     129.53
6              RR                       unknown       2     122.99
7              RR                  garden_tools       2     105.00
8              RR                   electronics       2      13.65
9              RR                    cool_stuff       2      64.99
10             RR                bed_bath_table       2     203.23
11             RR                 watches_gifts       1     219.00
12             RR                    stationery       1      39.99
13             RR                     perfumery       1     24

## Final Finding — SP Average Item Price Paradox (RESOLVED)

Observation: SP has highest total revenue (40,501 orders) but 
lowest avg item price (₹109) among all states.

Root Cause: Volume dilution — CONFIRMED as primary driver.
- Clear inverse relationship between order volume and avg price :
  exists across ALL 25 states consistently
- High order volume includes enormous diversity of purchases :
  the mass of small everyday items pulls the state average down
- This pattern holds without a single major exception in the data

Secondary Factor: Category mix — CONFIRMED as secondary driver.
- SP's top 3 categories (bed_bath_table ₹91,health_beauty ₹110, sports_leisure ₹104) are inherently lower-priced
- Smaller states like RR skew toward higher-priced categories :
  (home_appliances ₹318, baby ₹949) despite tiny order volumes

Hypothesis (unverifiable with current data): 
- SP's physical retail density means e-commerce is used for :
  convenience/low-cost purchases rather than big-ticket items
- Would need external demographic/retail data to confirm

Business Implication:
- Raw average item price is NOT a useful metric for comparing 
  states of different sizes — it is confounded by volume
- To compare true purchasing behavior, we should use 
  avg price WITHIN each product category, not overall avg price
- PE (₹144), CE (₹154), PA (₹165) have lower volumes but 
  meaningfully higher avg prices — could represent underserved, 
  high-intent markets worth targeting for expansion

In [11]:
# Cell 8 — Business Question B
# Q: Which product categories generate the most revenue?

result = pd.read_sql_query("""
    SELECT 
        product_category_name_english,
        COUNT(DISTINCT order_id) as total_orders,
        ROUND(SUM(price), 2) as total_revenue
    FROM orders_full
    GROUP BY product_category_name_english
    ORDER BY total_revenue DESC
    LIMIT 10
""", conn)
print("Q: Revenue by product category")
print(result)

Q: Revenue by product category
  product_category_name_english  total_orders  total_revenue
0                 health_beauty          8647     1233131.72
1                 watches_gifts          5495     1166176.98
2                bed_bath_table          9272     1023434.76
3                sports_leisure          7530      954852.55
4         computers_accessories          6530      888724.61
5               furniture_decor          6307      711927.69
6                    housewares          5743      615628.69
7                    cool_stuff          3559      610204.10
8                          auto          3810      578966.65
9                          toys          3804      471286.48


In [12]:
# Cell 9 — Business Question C
# Q: What is the average delivery delay by state? 
# (Negative = early, Positive = late)

result = pd.read_sql_query("""
    SELECT 
        customer_state,
        COUNT(DISTINCT order_id) as total_orders,
        ROUND(AVG(delivery_delay_days), 1) as avg_delay_days
    FROM orders_full
    GROUP BY customer_state
    HAVING total_orders > 100
    ORDER BY avg_delay_days DESC
    LIMIT 10
""", conn)
print("Q: Average delivery delay by state (states with 100+ orders)")
print(result)

Q: Average delivery delay by state (states with 100+ orders)
  customer_state  total_orders  avg_delay_days
0             AL           397            -8.7
1             MA           717            -9.9
2             SE           335           -10.0
3             ES          1995           -10.6
4             BA          3256           -11.0
5             CE          1279           -11.1
6             SP         40501           -11.2
7             MS           701           -11.2
8             PI           476           -11.5
9             SC          3546           -11.6


In [13]:
result = pd.read_sql_query("""
    SELECT 
        customer_state,
        COUNT(DISTINCT order_id) as total_orders,
        ROUND(AVG(delivery_delay_days), 1) as avg_delay_days
    FROM orders_full
    GROUP BY customer_state
    HAVING total_orders > 100
    ORDER BY avg_delay_days ASC
    LIMIT 10
""", conn)
print("States with worst delivery (most late):")
print(result)

States with worst delivery (most late):
  customer_state  total_orders  avg_delay_days
0             RO           243           -20.0
1             AM           145           -19.9
2             MT           886           -14.6
3             PA           946           -14.3
4             RS          5345           -14.1
5             RN           474           -14.0
6             PE          1593           -13.5
7             PR          4923           -13.5
8             MG         11354           -13.3
9             PB           517           -13.0


In [14]:
result = pd.read_sql_query("""
    SELECT 
        ROUND(AVG(delivery_delay_days), 1) as overall_avg_delay,
        COUNT(CASE WHEN delivery_delay_days > 0 THEN 1 END) as late_orders,
        COUNT(CASE WHEN delivery_delay_days <= 0 THEN 1 END) as early_orders,
        ROUND(COUNT(CASE WHEN delivery_delay_days > 0 THEN 1 END) * 100.0 
              / COUNT(*), 1) as pct_late
    FROM orders_full
""", conn)
print("Overall delivery performance:")
print(result)

Overall delivery performance:
   overall_avg_delay  late_orders  early_orders  pct_late
0              -12.0         7264        102925       6.6


In [15]:
result = pd.read_sql_query("""
    SELECT 
        CASE 
            WHEN delivery_delay_days > 0 THEN 'Late'
            WHEN delivery_delay_days BETWEEN -7 AND 0 THEN 'Slightly Early (0-7 days)'
            WHEN delivery_delay_days BETWEEN -14 AND -8 THEN 'Early (8-14 days)'
            ELSE 'Very Early (15+ days)'
        END as delivery_category,
        COUNT(DISTINCT order_id) as total_orders,
        ROUND(AVG(review_score), 2) as avg_review_score
    FROM orders_full
    WHERE review_score IS NOT NULL
    GROUP BY delivery_category
    ORDER BY avg_review_score DESC
""", conn)
print("Review score by delivery timing:")
print(result)

Review score by delivery timing:
           delivery_category  total_orders  avg_review_score
0          Early (8-14 days)         36158              4.24
1      Very Early (15+ days)         34779              4.22
2  Slightly Early (0-7 days)         18514              4.13
3                       Late          6381              2.26


In [16]:
result = pd.read_sql_query("""
    SELECT 
        CASE 
            WHEN delivery_delay_days > 0 THEN 'Late'
            WHEN delivery_delay_days BETWEEN -7 AND 0 
                THEN 'Slightly Early'
            WHEN delivery_delay_days BETWEEN -14 AND -8 
                THEN 'Early'
            ELSE 'Very Early'
        END as delivery_category,
        COUNT(DISTINCT order_id) as total_orders,
        ROUND(AVG(review_score), 2) as avg_review,
        ROUND(SUM(price), 2) as total_revenue
    FROM orders_full
    WHERE review_score IS NOT NULL
    GROUP BY delivery_category
    ORDER BY avg_review DESC
""", conn)
print(result)

  delivery_category  total_orders  avg_review  total_revenue
0             Early         36158        4.24     4757592.91
1        Very Early         34779        4.22     5013531.63
2    Slightly Early         18514        4.13     2385502.54
3              Late          6381        2.26      953366.55


In [17]:
result = pd.read_sql_query("""
    SELECT 
        CASE 
            WHEN delivery_delay_days > 0 THEN 'Late'
            WHEN delivery_delay_days BETWEEN -7 AND 0 
                THEN 'Slightly Early'
            WHEN delivery_delay_days BETWEEN -14 AND -8 
                THEN 'Early'
            ELSE 'Very Early'
        END as delivery_category,
        COUNT(DISTINCT order_id) as total_orders,
        ROUND(AVG(review_score), 2) as avg_review,
        ROUND(SUM(price), 2) as total_revenue,
        ROUND(AVG(price), 2) as avg_order_value
    FROM orders_full
    WHERE review_score IS NOT NULL
    GROUP BY delivery_category
    ORDER BY avg_review DESC
""", conn)
print(result)

  delivery_category  total_orders  avg_review  total_revenue  avg_order_value
0             Early         36158        4.24     4757592.91           115.69
1        Very Early         34779        4.22     5013531.63           124.52
2    Slightly Early         18514        4.13     2385502.54           114.13
3              Late          6381        2.26      953366.55           134.60


## EXECUTIVE FINDING — Late Delivery Disproportionately 
## Affects High-Value Customers

Avg order value by delivery timing:
Early (8-14 days)      → ₹115.69 avg order value   4.24★
Very Early (15+ days)  → ₹124.52 avg order value   4.22★
Slightly Early (0-7)   → ₹114.13 avg order value   4.13★
Late                   → ₹134.60 avg order value   2.26★  ← HIGHEST

Counter-intuitive finding:
Late orders have the HIGHEST average order value of all 
four delivery categories — meaning the platform's worst 
delivery experience is disproportionately falling on its 
highest-spending customers.

Possible explanations:
1. Heavy/bulky high-value items (furniture, appliances, 
   electronics) are harder to deliver on time due to 
   logistics complexity — and also cost more
2. High-value orders may travel longer distances to reach 
   customers in underserved areas (consistent with our 
   finding that remote states have lower order volumes 
   but higher avg prices)

Business Implication:
This is not a "small customer" problem. The 6,381 late 
orders represent the platform's highest-spending segment experiencing a 2.26★ satisfaction level.

Priority recommendation: Investigate whether specific 
product categories (furniture, appliances, heavy goods) 
account for a disproportionate share of late deliveries 

if so, category-specific logistics SLAs would address 
the root cause directly.

Revised revenue at risk: ₹9,53,366 from highest-spending 
customers — actual lifetime value loss significantly 
higher given zero probability of reorder at 2.26★.